# Topic 4 그림 코드 모음 — 강의용 Colab 데모

**확률통계 · Topic 4 · A Zoo of Discrete Distributions**

슬라이드 [T04_slides_v2.md](slides/T04_slides_v2.md) 에 쓴 그림 3개를 만드는 코드를 모았다.
슬라이드는 고정된 PNG지만, 이 노트북은 **강의 중 숫자를 바꿔가며 즉석에서 다시 그려볼 수 있다.**

| 그림 | 슬라이드 | 원본 스크립트 |
|---|---|---|
| ① 서버 로그 → Poisson 적합 | `[S]` 서버 로그를 그냥 그려 본다 | `figs_src_v2/t04v2_server_log.py` |
| ② 이산 분포 네 개를 한 화면에 | `[C]` 분포는 상황이 정한다 | `figs_src_v2/t04v2_zoo.py` |
| ③ Binomial → Poisson 수렴 | `[C]` Poisson의 정체 | `figs_src_v2/t04v2_binom_poisson.py` |

⚠️ 실제 슬라이드 그림은 Windows 로컬에서 `Malgun Gothic`으로 렌더했다. 이 노트북은 Colab(Linux)이라
나눔고딕을 대신 설치해서 쓴다 — 글꼴만 다르고 배치·색·수치는 슬라이드와 동일하다.

**맨 위 설정 셀을 한 번 실행한 뒤, 그림 셀은 순서와 상관없이 원하는 것만 실행하면 된다.**

In [ ]:
# ── 설정: 한글 글꼴 + 스타일 (맨 처음 한 번만 실행) ─────────────────────
# Colab은 Linux라 한글 글꼴이 기본으로 없다. 나눔고딕을 설치해 등록한다.
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import binom, geom, poisson

try:
    for f in fm.findSystemFonts(fontpaths=["/usr/share/fonts/truetype/nanum"]):
        fm.fontManager.addfont(f)
    KOREAN_FONT = "NanumGothic"
except Exception:
    KOREAN_FONT = "DejaVu Sans"  # 설치 실패 시 — 한글은 깨지지만 그림은 그려진다
    print("나눔고딕 설치/등록 실패 — DejaVu Sans로 대신한다 (한글이 깨질 수 있음)")

# 슬라이드 테마와 같은 색 (assets/deckstyle.py 와 동일)
C = {
    "accent": "#3b4fd8", "teal": "#0d9488", "orange": "#d97d17",
    "purple": "#8b5cf6", "pink": "#d9457f", "ink": "#0f172a",
    "body": "#4b5768", "muted": "#97a3b6", "line": "#e6eaf1", "soft": "#f7f9fc",
}
CYCLE = [C["accent"], C["teal"], C["orange"], C["purple"], C["pink"], C["muted"]]

mpl.rcParams.update({
    "font.family": KOREAN_FONT,
    "mathtext.fontset": "dejavusans",
    "axes.unicode_minus": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.titleweight": "bold",
    "axes.labelsize": 13.5,
    "xtick.labelsize": 12.5,
    "ytick.labelsize": 12.5,
    "legend.fontsize": 12.5,
    "legend.frameon": False,
    "lines.linewidth": 2.0,
    "axes.prop_cycle": mpl.cycler(color=CYCLE),
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.color": C["line"],
    "grid.linewidth": 1.0,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#d7dce6",
    "figure.dpi": 110,
})

## ① 서버 로그 → Poisson 적합

`data/t04_server_log.csv` — 하루치(1,440분) 서버 접속 로그. 왼쪽은 원자료, 오른쪽은
10~15시만 떼어낸 히스토그램에 Poisson을 겹친 것이다. 강의 중엔 `10, 15` 시간대를
바꿔가며 "구간을 좁히면 왜 Poisson에 더 잘 맞는가"를 보여줄 수 있다.

In [ ]:
import os

URL = "https://inetguru.github.io/prob-stat-pnuace/data/t04_server_log.csv"
src = "t04_server_log.csv" if os.path.exists("t04_server_log.csv") else URL
log = pd.read_csv(src)
day = log[(log.hour >= 10) & (log.hour < 15)]["requests"].to_numpy()

m_all, v_all = log["requests"].mean(), log["requests"].var(ddof=0)
m, v = day.mean(), day.var(ddof=0)
print(f"하루 전체  평균 {m_all:.2f}  분산 {v_all:.2f}  분산/평균 {v_all / m_all:.2f}")
print(f"10~15시    평균 {m:.2f}  분산 {v:.2f}  분산/평균 {v / m:.2f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.6, 3.5),
                               gridspec_kw={"width_ratios": [1.35, 1]})

ax1.plot(log["minute"], log["requests"], color=C["accent"], linewidth=0.7, alpha=0.85)
ax1.axvspan(10 * 60, 15 * 60, color=C["orange"], alpha=0.13)
ax1.text(12.5 * 60, log["requests"].max() * 0.96, "10~15시",
         ha="center", va="top", fontsize=13, fontweight="bold", color=C["orange"])
ax1.set_title("하루치 원자료 — 1분마다 센 요청 수", color=C["ink"])
ax1.set_xlabel("시각 (시)")
ax1.set_ylabel("요청 수")
ax1.set_xticks([0, 360, 720, 1080, 1440])
ax1.set_xticklabels(["0", "6", "12", "18", "24"])

ks = np.arange(0, day.max() + 2)
counts = np.array([(day == k).sum() for k in ks])
ax2.bar(ks, counts / counts.sum(), width=0.72, color=C["teal"],
        alpha=0.85, edgecolor="white", label="관측 (10~15시)")
ax2.plot(ks, poisson.pmf(ks, m), "o-", color=C["orange"], linewidth=2.0,
         markersize=5, label=f"Poisson($\\lambda$={m:.2f})")
ax2.set_title(f"평균 {m:.2f} · 분산 {v:.2f}", color=C["ink"])
ax2.set_xlabel("1분당 요청 수")
ax2.set_ylabel("비율")
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

## ② 이산 분포 네 개를 한 화면에

제목은 공식이 아니라 **생성 스토리**를 적어 둔 것이다 — 분포는 외우는 게 아니라
"어떤 상황에서 생기는가"로 고른다는 슬라이드의 요지를 그대로 옮겼다.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12.0, 3.0))

# ① Bernoulli(0.3) — 한 번 시도했다. 성공? 실패?
p = 0.3
axes[0].bar([0, 1], [1 - p, p], width=0.5, color=C["accent"], alpha=0.85,
            edgecolor="white")
axes[0].set_title("Bernoulli(0.3)\n한 번 시도 — 성공? 실패?", color=C["ink"], fontsize=13)
axes[0].set_xticks([0, 1])

# ② Binomial(20, 0.3) — n번 시도해서 성공이 몇 번?
n, p = 20, 0.3
k = np.arange(0, n + 1)
axes[1].bar(k, binom.pmf(k, n, p), width=0.72, color=C["teal"], alpha=0.85,
            edgecolor="white")
axes[1].set_title("Binomial(20, 0.3)\n20번 시도 — 성공이 몇 번?", color=C["ink"], fontsize=13)

# ③ Geometric(0.3) — 첫 성공까지 몇 번?
k = np.arange(1, 16)
axes[2].bar(k, geom.pmf(k, 0.3), width=0.72, color=C["orange"], alpha=0.85,
            edgecolor="white")
axes[2].set_title("Geometric(0.3)\n첫 성공까지 몇 번?", color=C["ink"], fontsize=13)

# ④ Poisson(8) — 정해진 구간에서 몇 번?
k = np.arange(0, 21)
axes[3].bar(k, poisson.pmf(k, 8), width=0.72, color=C["purple"], alpha=0.85,
            edgecolor="white")
axes[3].set_title("Poisson(8)\n정해진 시간에 몇 번?", color=C["ink"], fontsize=13)

for ax in axes:
    ax.set_yticks([])
    ax.grid(False)
    ax.spines["left"].set_visible(False)
axes[0].set_ylabel("확률", color=C["body"])

fig.subplots_adjust(wspace=0.22)
print("Bernoulli E=0.30 · Binomial E=%.1f · Geometric E=%.2f · Poisson E=8"
      % (20 * 0.3, 1 / 0.3))
plt.show()

## ③ Binomial → Poisson 수렴

$np = 3$ 을 고정하고 $n$ 을 키우면 Binomial$(n, 3/n)$ 이 Poisson$(3)$ 에 점점 겹쳐진다.
`NS` 리스트의 숫자를 바꿔가며 수렴 속도를 즉석에서 보여줄 수 있다.

In [ ]:
LAM = 3
NS = [10, 50, 500]
COLORS = [C["orange"], C["teal"], C["accent"]]

k = np.arange(0, 13)
pois = poisson.pmf(k, LAM)

fig, axes = plt.subplots(1, 3, figsize=(11.6, 3.3), sharey=True)

for ax, n, col in zip(axes, NS, COLORS):
    p = LAM / n
    b = binom.pmf(k, n, p)
    gap = np.abs(b - pois).max()
    print(f"n={n:>4}  p={p:.4f}  최대 차이 {gap:.4f}")

    ax.bar(k, b, width=0.66, color=col, alpha=0.85, edgecolor="white",
           label=f"Binomial({n}, {p:.3f})")
    ax.plot(k, pois, "o--", color=C["ink"], linewidth=1.8, markersize=4.5,
            label="Poisson(3)")
    ax.set_title(f"n = {n}  ·  최대 차이 {gap:.4f}", color=C["ink"], fontsize=14)
    ax.set_xlabel("k")
    ax.legend(loc="upper right", fontsize=11)

axes[0].set_ylabel("확률")
fig.subplots_adjust(wspace=0.12)
plt.show()

---
이 노트북은 채점 대상이 아니다. 슬라이드 그림을 재생성하는 [figs_src_v2/](figs_src_v2/) 스크립트가
원본이며, 슬라이드 PNG를 바꾸려면 그쪽을 고치고 다시 실행해야 한다 — 이 노트북은 강의 중
라이브 데모·질의응답용 사본이다.